# Pokémon Name Generator (PyTorch RNN)
Implementation of a character-level language model capable of generating plausible Pokémon names

In [1]:
import torch
from utils import load_device

In [2]:
device=load_device()

PyTorch Version: 2.8.0+cu126
Using GPU: NVIDIA GeForce GTX 1660 Ti


In [3]:
import numpy as np 
import pandas as pd
df=pd.read_csv('datasets/pokemon_complete.csv')

In [4]:
names=df['name'].str.lower()

In [5]:
names = names.str.split('-mega').str[0]

In [6]:
mask=names.str.contains('-')

In [7]:
replacement=names[mask].iloc[-231:].str.split('-').str[0]
names[replacement.index]=replacement

In [8]:
attribute_pattern = r'-female|-male|-incarnate|-average|-curly|iron-|-two-segment'
attr_mask = names.str.contains(attribute_pattern, regex=True)
names.loc[attr_mask] = names[attr_mask].str.split('-').str[0]

In [9]:
names = names.str.replace('-', ' ')

In [10]:
names = names.str.split(r'\d').str[0]

In [11]:
names = names.str.strip().drop_duplicates()

In [12]:
sos='@'
eos='#'
pad='$'

In [13]:
names_list=names.values.tolist()
vocab=sorted(set("".join(names_list)))
vocab=[pad,sos,eos]+vocab

In [14]:
"".join(vocab)

'$@# abcdefghijklmnopqrstuvwxyz'

In [15]:
def add_special_tokens(names,sos,eos,pad):
    max_len=max(names.str.len())
    diff=max_len-names.str.len()
    names=sos+names+eos  
    names+=diff.apply(lambda n:n*pad)
    return names

In [16]:
names=add_special_tokens(names,sos,eos,pad)

In [17]:
idx_to_char={i:c for i,c in enumerate(vocab)}
char_to_idx={c:i for i,c in enumerate(vocab)}

In [18]:
def encode_name(name,char_to_idx):
    return torch.tensor([char_to_idx[c]for c in name])
def decode_name(ids,idx_to_char): 
    return "".join([idx_to_char[i.item()]for i in ids])

In [19]:
encoded = encode_name("@hello world#$",char_to_idx)
encoded

tensor([ 1, 11,  8, 15, 15, 18,  3, 26, 18, 21, 15,  7,  2,  0])

In [20]:
decode_name(encoded,idx_to_char)

'@hello world#$'

In [21]:
import torch.nn.functional as F
def next_char(model, text,char_to_idx ,idx_to_char,temperature=1):
    encoded_text = encode_name(text,char_to_idx).unsqueeze(dim=0).to(device)
    model.eval()
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return idx_to_char[predicted_char_id]

In [22]:
def generate_name(model,text,sos,char_to_idx ,idx_to_char, n_chars=50, temperature=1):
    text=sos+text
    for _ in range(n_chars):
        char = next_char(model, text,char_to_idx ,idx_to_char, temperature)
        if char==eos:
            break 
        text+=char
    return text[1:]

In [23]:
def generate_n_names(model,n,text,sos,char_to_idx ,idx_to_char, n_chars=50, temperature=1):
    generated_names=[]
    for _ in range(n):
        generated_names.append(generate_name(model,text,sos,char_to_idx ,idx_to_char, n_chars=n_chars, temperature=temperature))
    return generated_names

In [24]:
from torch.utils.data import Dataset, DataLoader
class CharDataset(Dataset):
    def __init__(self, names, window_length, char_to_idx):
        self.names = names
        self.window_length = window_length
        self.char_to_idx = char_to_idx

    def __len__(self):
        return len(self.names)
    def __getitem__(self,idx): 
        if idx >= len(self): 
            raise IndexError("dataset index out of range")
        name = self.names[idx]
        encoded = encode_name(name, self.char_to_idx)
        X = encoded[:self.window_length]
        y = encoded[1:self.window_length + 1]
        
        return X, y

In [25]:
sample_encoded = encode_name(names.iloc[0], char_to_idx)
sequence_length = len(sample_encoded) - 1 
dataset = CharDataset(names=names.values.tolist(), window_length=sequence_length, char_to_idx=char_to_idx)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True) 

In [26]:
def train(model, optimizer, loss_fn, metric, train_loader,
          n_epochs,device, track_valid=True,valid_loader=None, early_stopping=False, patience=10,
          checkpoint_path=None, scheduler=None, mode="max"):
    if track_valid and valid_loader is None:
        raise ValueError("valid_loader is None with track_valid=True")
    if early_stopping:
        track_valid = True
        
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    
    best_metric = -float('inf') if mode == "max" else float('inf')
    patience_counter = 0
    
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        t0 = time.time()
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)#change to stabilize training
            optimizer.step()
            
            total_loss += loss.item()
            metric.update(y_pred, y_batch)
            
        t1 = time.time()
        train_metric = metric.compute().item()
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        
        summary = (f"Epoch {epoch + 1}/{n_epochs}, "
                   f"train loss: {history['train_losses'][-1]:.4f}, "
                   f"train metric: {history['train_metrics'][-1]:.4f}, ")
        
        if track_valid:
            valid_metric = evaluate(model, valid_loader, metric,device).item()
            history["valid_metrics"].append(valid_metric)
            
            is_best = (valid_metric > best_metric) if mode == "max" else (valid_metric < best_metric)
            
            if is_best:
                torch.save(model.state_dict(), checkpoint_path)
                best_metric = valid_metric
                best_tag = " (best)"
                patience_counter = 0
            else:
                best_tag = ""
                patience_counter += 1
                
            summary += f"valid metric: {history['valid_metrics'][-1]:.4f}{best_tag}"
            
        print(summary + f" in {t1 - t0:.1f}s")
        
        if scheduler is not None:
            scheduler.step()
                
        if early_stopping and patience_counter >= patience:
            print("Early stopping triggered")
            break
    if track_valid and os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        
    return history

In [27]:
import torch.nn as nn
class PokemonModel(nn.Module): 
    def __init__(self,vocab_size,n_layers=2,embed_dim=32,hidden_dim=128,dropout=0.5):
        super().__init__()
        self.embed=nn.Embedding(vocab_size,embed_dim)
        self.gru=nn.GRU(embed_dim,hidden_dim,num_layers=n_layers,batch_first=True,dropout=dropout)
        self.output=nn.Linear(hidden_dim,vocab_size)
        self.output
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)
    def forward(self,X): 
        embeddings=self.embed(X)
        outputs,_states=self.gru(embeddings)
        return self.output(outputs).permute(0,2,1)

In [28]:
import torchmetrics
torch.manual_seed(42)
n_epochs=70
vocab_size=len(vocab)
model=PokemonModel(vocab_size,hidden_dim=64).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-2,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)
xentropy=nn.CrossEntropyLoss(ignore_index=char_to_idx[pad])
accuracy = torchmetrics.Accuracy(task="multiclass",num_classes=vocab_size,ignore_index=char_to_idx[pad]).to(device)

In [29]:
generate_n_names(model,10,"",sos,char_to_idx ,idx_to_char)

['s$fq@glhvsdowmyblmzd',
 'o$bntfhjkfofmxnflgcy',
 'gun scy',
 'qtrmtylnydwjffotufcxnlflkvrnr y mjl',
 'pyfypjnlysenprdknjhcwtiqs$sie ynaxqtakwnlohbxwrorb',
 'ayzewafxeqmqo$kcmwvxoonghqogwggkcthwdq@lmowwowoolg',
 'adxzbkkxnnwif@cygmcsasj@ofcmhwm$sdy uhi y qcok iyh',
 'bi chdmmiodztrwgn hpm kmpgxwdh',
 ' xyhad@xntukiwx onwahdx hwzx$$g u@$pl @pbhreasnkc$',
 'bgmj@ xdj$qfbttmihcupdhjmiarigenvxwqrbmlxu']

In [30]:
from utils import train
history=train(model,optimizer,xentropy,accuracy,dataloader,n_epochs,device,track_valid=False,scheduler=scheduler)

Epoch 1/70, train loss: 2.8849, train metric: 0.1770,  in 0.4s
Epoch 2/70, train loss: 2.6348, train metric: 0.2114,  in 0.2s
Epoch 3/70, train loss: 2.5484, train metric: 0.2236,  in 0.2s
Epoch 4/70, train loss: 2.5101, train metric: 0.2286,  in 0.1s
Epoch 5/70, train loss: 2.4635, train metric: 0.2425,  in 0.2s
Epoch 6/70, train loss: 2.4364, train metric: 0.2519,  in 0.2s
Epoch 7/70, train loss: 2.4105, train metric: 0.2504,  in 0.2s
Epoch 8/70, train loss: 2.3821, train metric: 0.2617,  in 0.2s
Epoch 9/70, train loss: 2.3714, train metric: 0.2648,  in 0.1s
Epoch 10/70, train loss: 2.3458, train metric: 0.2776,  in 0.2s
Epoch 11/70, train loss: 2.3276, train metric: 0.2734,  in 0.2s
Epoch 12/70, train loss: 2.3079, train metric: 0.2805,  in 0.2s
Epoch 13/70, train loss: 2.2857, train metric: 0.2906,  in 0.1s
Epoch 14/70, train loss: 2.2733, train metric: 0.2944,  in 0.2s
Epoch 15/70, train loss: 2.2692, train metric: 0.2947,  in 0.2s
Epoch 16/70, train loss: 2.2431, train metric: 0.

In [32]:
generate_n_names(model,20,"",sos,char_to_idx ,idx_to_char,temperature=0.6)

['starubuff',
 'latios',
 'dravile',
 'pincack',
 'phypire',
 'groslee',
 'coledor',
 'arctuza',
 'dingelg',
 'druvele',
 'magneruma',
 'haushord',
 'chi bent',
 'coscalia',
 'skitty',
 'shellett',
 'corchreen',
 'hattren',
 'shitneor',
 'duskull']